# Phase 1 indicators: FARN reconciliation + function-level real / %GDP / share / RRI

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\Juan Ignacio\.antigravity\projects\milei-climate-narratives\analysis\D2_budget"
)

PROCESSED_DIR = PROJECT_DIR / "processed"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

ipc = pd.read_csv(
    PROCESSED_DIR / "ipc_anual_promedio.csv",
    index_col="anio",
)["ipc"]

# OJO ACÁ VER IPC PROMEDIO SI ESTÁ OK. Al chequear con FARN daría igual que ellos.

fn = pd.read_csv(
    PROCESSED_DIR / "panel_funcion.csv"
)

# nominal GDP at market prices, millones de pesos corrientes.
# Source: INDEC national accounts via apis.datos.gob.ar, VAB a precios basicos
# (8.1_VABPB_2004_A_36) + impuestos netos sobre los productos (8.1_IVA_2004_A_23).
# Written by scripts/03_robustness_2026_and_gdp.py -> processed/pib_nominal_indec.csv

In [4]:
_p=pd.read_csv(PROCESSED_DIR / "pib_nominal_indec.csv")
PIB=dict(zip(_p.anio,_p.pib_mm_pesos))

MEAS={2023:'credito_devengado',2024:'credito_devengado',2025:'credito_devengado',
      2026:'credito_vigente'}
fn['monto']=[r[MEAS[r['anio']]] for _,r in fn.iterrows()]

def wide(keys):
    w=fn.groupby(['anio']+keys,as_index=False)['monto'].sum()
    return w.pivot_table(index=keys,columns='anio',values='monto').fillna(0.0)

def real_chg(w,y,base=2023):
    return (w[y]/ipc[y])/(w[base]/ipc[base])-1

## 1. FARN reconciliation (finalidad level)

In [6]:
f=wide(['finalidad_id','finalidad_desc'])
tot=fn.groupby('anio')['monto'].sum()
farn={'ADMINISTRACION GUBERNAMENTAL':(2349060,6652753,-0.376),
      'DEUDA PUBLICA':(3472801,10352974,-0.343),
      'SERVICIOS DE DEFENSA Y SEGURIDAD':(1710842,6075017,-0.217),
      'SERVICIOS ECONOMICOS':(5647579,9539839,-0.628),
      'SERVICIOS SOCIALES':(24845207,90257189,-0.199)}
rows=[]
for (fid,fd),r in f.iterrows():
    a,b,c=farn[fd]
    rows.append(dict(finalidad=fd, ours_2023=r[2023], farn_2023=a, d23=r[2023]/a-1,
                     ours_2025=r[2025], farn_2025=b, d25=r[2025]/b-1,
                     ours_real=real_chg(f,2025).loc[(fid,fd)], farn_real=c))
rec=pd.DataFrame(rows)
rec.loc[len(rec)]=dict(finalidad='TOTAL',ours_2023=tot[2023],farn_2023=38025489,
    d23=tot[2023]/38025489-1, ours_2025=tot[2025],farn_2025=122877771,
    d25=tot[2025]/122877771-1,
    ours_real=(tot[2025]/ipc[2025])/(tot[2023]/ipc[2023])-1, farn_real=-0.288)
rec.to_csv(OUTPUTS_DIR / "farn_reconciliation.csv",index=False)
print(rec.to_string(index=False,float_format=lambda x:f'{x:,.4f}'))

                       finalidad       ours_2023  farn_2023     d23        ours_2025  farn_2025    d25  ours_real  farn_real
    ADMINISTRACION GUBERNAMENTAL  2,349,060.4746    2349060  0.0000   6,728,915.6600    6652753 0.0114    -0.3690    -0.3760
SERVICIOS DE DEFENSA Y SEGURIDAD  1,710,841.6113    1710842 -0.0000   6,096,088.9215    6075017 0.0035    -0.2151    -0.2170
              SERVICIOS SOCIALES 24,845,207.0713   24845207  0.0000  90,394,589.8256   90257189 0.0015    -0.1986    -0.1990
            SERVICIOS ECONOMICOS  5,647,579.0778    5647579  0.0000   9,960,289.8108    9539839 0.0441    -0.6115    -0.6280
                   DEUDA PUBLICA  3,472,800.8352    3472801 -0.0000  10,354,070.7958   10352974 0.0001    -0.3433    -0.3430
                           TOTAL 38,025,489.0701   38025489  0.0000 123,533,955.0137  122877771 0.0053    -0.2844    -0.2880


Farn results are reconcilled

## 2. Function-level table

In [9]:
g=wide(['finalidad_id','funcion_id','funcion_desc'])
res=pd.DataFrame(index=g.index)
res['dev_2023']=g[2023]; res['dev_2024']=g[2024]; res['dev_2025']=g[2025]; res['vig_2026']=g[2026]
res['real_24_23']=real_chg(g,2024)
res['real_25_23']=real_chg(g,2025)
res['real_26_23']=real_chg(g,2026)
for y in [2023,2024,2025,2026]:
    res[f'share_{y}']=g[y]/tot[y]*100
res['pibshare_2023']=g[2023]/PIB[2023]*100
res['pibshare_2024']=g[2024]/PIB[2024]*100
res['pibshare_2025']=g[2025]/PIB[2025]*100
res['pibshare_chg_25_23']=res['pibshare_2025']/res['pibshare_2023']-1
tot_real25=(tot[2025]/ipc[2025])/(tot[2023]/ipc[2023])-1
tot_real24=(tot[2024]/ipc[2024])/(tot[2023]/ipc[2023])-1
res['RRI_25']=(res['real_25_23']-tot_real25)*100   # percentage points vs total
res['RRI_24']=(res['real_24_23']-tot_real24)*100
res['priority_ratio_25']=res['share_2025']/res['share_2023']
res=res[res['dev_2023']>0].sort_values('real_25_23')
res.to_csv(OUTPUTS_DIR / "panel_funcion_indicadores.csv")
print(f"\nTOTAL real 2024/2023 {tot_real24:+.3%}  |  2025/2023 {tot_real25:+.3%}")
print(res[['dev_2023','dev_2025','real_25_23','pibshare_chg_25_23','share_2023','share_2025','RRI_25']]
      .to_string(float_format=lambda x:f'{x:,.3f}'))


TOTAL real 2024/2023 -25.847%  |  2025/2023 -28.440%
                                                                  dev_2023       dev_2025  real_25_23  pibshare_chg_25_23  share_2023  share_2025  RRI_25
finalidad_id funcion_id funcion_desc                                                                                                                     
3            7          Vivienda y Urbanismo                   335,090.799     30,661.645      -0.980              -0.979       0.881       0.025 -69.545
             8          Agua Potable y Alcantarillado          423,228.439    165,199.074      -0.914              -0.911       1.113       0.134 -62.963
4            6          Industria                              121,036.298    103,163.602      -0.812              -0.805       0.318       0.084 -52.786
             7          Comercio, Turismo y Otros Servicios     81,244.305     70,170.529      -0.810              -0.803       0.214       0.057 -52.536
             2        

### Visualización tabla para exploración visual

In [10]:
explore_funciones = (
    res
    .reset_index()
    [
        [
            'finalidad_id',
            'funcion_id',
            'funcion_desc',
            'dev_2023',
            'dev_2025',
            'real_25_23',
            'share_2023',
            'share_2025',
            'pibshare_chg_25_23',
            'RRI_25',
            'priority_ratio_25'
        ]
    ]
)

explore_funciones

,finalidad_id,funcion_id,funcion_desc,dev_2023,dev_2025,real_25_23,share_2023,share_2025,pibshare_chg_25_23,RRI_25,priority_ratio_25
0,3,7,Vivienda y Urbanismo,3.350908e+05,3.066164e+04,-0.979844,0.881227,0.024820,-0.979082,-69.544943,0.028166
1,3,8,Agua Potable y Alcantarillado,4.232284e+05,1.651991e+05,-0.914021,1.113012,0.133728,-0.910768,-62.962555,0.120149
2,4,6,Industria,1.210363e+05,1.031636e+05,-0.812253,0.318303,0.083510,-0.805151,-52.785813,0.262361
3,4,7,"Comercio, Turismo y Otros Servicios",8.124430e+04,7.017053e+04,-0.809750,0.213657,0.056803,-0.802553,-52.535544,0.265858
4,4,2,Comunicaciones,1.883220e+05,1.689883e+05,-0.802341,0.495252,0.136795,-0.794863,-51.794564,0.276213
5,3,2,Promoción y Asistencia Social,2.477261e+06,2.912423e+06,-0.741033,6.514738,2.357589,-0.731236,-45.663798,0.361885
6,4,4,Ecología y Desarrollo Sostenible,1.192548e+05,1.913703e+05,-0.646523,0.313618,0.154913,-0.633152,-36.212848,0.493955
7,4,5,"Agricultura, Ganadería y Pesca",1.947912e+05,3.301202e+05,-0.626694,0.512265,0.267230,-0.612573,-34.229937,0.521664
8,1,3,Dirección Superior Ejecutiva,1.797993e+05,3.056942e+05,-0.625492,0.472839,0.247458,-0.611325,-34.109706,0.523344
9,4,1,"Energía, Combustibles y Minería",3.204959e+06,5.770587e+06,-0.603394,8.428449,4.671255,-0.588391,-31.899897,0.554225


### Results interpretation
- Trabajo = +106.5%
Decomposing función 3.6 shows one line driving everything: Acciones de Empleo goes from 46,704 to 946,701, and inside it sits a single activity, "Volver al Trabajo" at 891,479 millones (80.8% of the whole function).

Further explained in D2_phase1_funcion_results